<a href="https://colab.research.google.com/github/Hiten1896/RAG-Document-Agent/blob/main/rag_agent_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import shutil
import traceback
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

# Load environment variables from .env
load_dotenv()

app = FastAPI(title="DocAgent RAG Backend")

# Enable CORS for local Next.js frontend and deployed Vercel domains
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # Allows local dev and production Vercel frontend
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Persistent directory for ChromaDB
CHROMA_DIR = "./chroma_db"

# Initialize lightweight Google Gemini Embedding API (cuts RAM usage to ~120MB)
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")


@app.get("/")
async def root():
    """Health check endpoint to prevent 404 errors on root GET requests."""
    return {"status": "FastAPI Backend Active", "message": "DocAgent RAG Service is running"}


@app.post("/upload")
async def upload_pdf(file: UploadFile = File(...)):
    """Handles PDF uploading, text splitting, embedding generation, and Chroma storage."""
    temp_file_path = f"temp_{file.filename}"
    
    try:
        # 1. Save uploaded file temporarily to disk
        with open(temp_file_path, "wb") as buffer:
            shutil.copyfileobj(file.file, buffer)

        # 2. Extract text using PyPDFLoader
        loader = PyPDFLoader(temp_file_path)
        docs = loader.load()

        if not docs:
            raise HTTPException(status_code=400, detail="The PDF file appears to be empty.")

        # 3. Chunk text into manageable pieces
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
        chunks = text_splitter.split_documents(docs)

        # Filter out empty or whitespace-only chunks
        valid_chunks = [c for c in chunks if c.page_content and c.page_content.strip()]

        if not valid_chunks:
            raise HTTPException(
                status_code=400, 
                detail="Could not extract readable text. The PDF might contain only scanned images."
            )

        print(f"DEBUG: Processing {len(valid_chunks)} text chunk(s) for file: {file.filename}")

        # 4. Store/Upsert chunks into Chroma VectorStore using Gemini Embeddings
        vector_store = Chroma.from_documents(
            documents=valid_chunks,
            embedding=embeddings,
            persist_directory=CHROMA_DIR
        )

        return {
            "message": "File processed successfully",
            "filename": file.filename,
            "chunks_processed": len(valid_chunks)
        }

    except Exception as e:
        print("\n=== UPLOAD PROCESSING ERROR ===")
        traceback.print_exc()
        print("=================================\n")
        raise HTTPException(status_code=500, detail=f"Failed to process file: {str(e)}")

    finally:
        # Clean up temporary PDF file from disk
        if os.path.exists(temp_file_path):
            os.remove(temp_file_path)


if __name__ == "__main__":
    import uvicorn
    # Bind to Render's dynamic PORT variable or fallback to 8000
    port = int(os.environ.get("PORT", 8000))
    uvicorn.run("main:app", host="0.0.0.0", port=port, reload=False)